# Alia on a Colab T4: 35B Q4 with 16 GB VRAM

Measures Qwen3.6-35B-A3B **UD-Q4_K_XL** (22.4 GB) on a free Colab T4 (16 GB VRAM) with the llama.cpp build Alia pins
(b11639 = a0e6dc7), at the context of the `rtx12-16gb-qwen36-35b-q4` profile (32,768).

**Variant, not the exact profile.** The profile needs 30 GB RAM; Colab free has 12.7 GB. With the profile's exact
`server_args` the server is OOM-killed while loading (exit -9). This notebook adds `--no-repack --no-host`, so the
expert weights that do not fit in VRAM stay memory-mapped from disk instead of being copied into RAM. Results are
labelled `colab-t4-12-16gb-norepack`.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Every long step prints a
status line every 20–30 seconds. Download ≈ 10–20 min, measurement ≈ 15–25 min. If a cell stops with an error,
run that cell again: every step resumes where it stopped. Copy the output of the last cell.

## 1. Machine (stops leftover servers from earlier runs)

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || { echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"; exit 1; }
pkill -x llama-server && echo "stopped an old llama-server" || true
free -g | head -2
df -h /content | tail -1

## 2. llama.cpp b11639 (CUDA 12.8 release build)

In [ ]:
%%bash
set -e
cd /content
if [ ! -x llama/llama-b11639/llama-server ] && [ -z "$(find /content/llama -name llama-server -type f 2>/dev/null)" ]; then
  R=https://github.com/ggml-org/llama.cpp/releases/download/b11639
  for f in llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz cudart-llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz; do
    echo "downloading $f"; wget -q -c $R/$f; mkdir -p llama; tar xzf $f -C llama
  done
fi
BIN=$(find /content/llama -name llama-server -type f | head -1)
echo "$BIN"
LD_LIBRARY_PATH=$(dirname $BIN) $BIN --version 2>&1 | tail -2

## 3. Model: 22.4 GB, plain HTTP download straight to disk (resumable)

In [ ]:
%%bash
F=/content/models/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf
SIZE=22360456160
URL=https://huggingface.co/unsloth/Qwen3.6-35B-A3B-GGUF/resolve/main/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf
mkdir -p /content/models && rm -rf /content/models/.cache
if [ "$(stat -c %s $F 2>/dev/null)" != "$SIZE" ]; then
  wget -q -c -O $F "$URL" &
  P=$!
  while kill -0 $P 2>/dev/null; do
    sleep 20
    HAVE=$(stat -c %s $F 2>/dev/null || echo 0)
    echo "$(date +%T)  $((HAVE * 100 / SIZE)) %  $((HAVE / 1000000000)) of 22 GB   RAM used $(free -g | awk '/Mem/{print $3"/"$2}') GB"
  done
  wait $P
fi
if [ "$(stat -c %s $F)" = "$SIZE" ]; then echo "model complete"; else echo "download incomplete: run this cell again"; exit 1; fi

## 4. Alia repo (bench scripts)

In [ ]:
%%bash
cd /content
[ -d alia ] || git clone -q --branch micro/r10 https://github.com/BEKO2210/alia
cd alia && git pull -q && git log --oneline -1

## 5. Measure: 3 reps at 4K, 16K and 28K tokens of context

In [ ]:
%%bash
cd /content/alia
export LLAMA_SERVER=$(find /content/llama -name llama-server -type f | head -1)
LOG=bench/results/llama-server-colab-t4-12-16gb-norepack-fit.log
rm -f $LOG
# status line every 30 s: RAM, VRAM and the last line of the server log
( while sleep 30; do
    echo "[$(date +%T)] RAM $(free -g | awk '/Mem/{print $3"/"$2}') GB | VRAM $(nvidia-smi --query-gpu=memory.used --format=csv,noheader) | server: $(tail -n 1 $LOG 2>/dev/null | cut -c1-80)"
  done ) &
MON=$!
# profile server_args of rtx12-16gb-qwen36-35b-q4 + --no-repack --no-host (see top); no -ngl, llama.cpp --fit decides
python3 -u bench/bench_llamacpp.py --gguf /content/models/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf \
  --name colab-t4-12-16gb-norepack --configs fit --depths 4096,16384,28000 --reps 3 --ctx 32768 \
  --extra "-b 2048 -ub 2048 --jinja --cache-ram 2048 --no-repack --no-host" 2>&1
kill $MON

## 6. Result: copy everything below

In [ ]:
import json, glob, os, re
os.chdir("/content/alia")
rows = []
for f in sorted(glob.glob("bench/results/llamacpp-*.jsonl"), key=os.path.getmtime):
    rows += [r for r in map(json.loads, open(f)) if r.get("model") == "colab-t4-12-16gb-norepack"]
if not rows:
    print("No result rows. Server log tail:")
    print("".join(open("bench/results/llama-server-colab-t4-12-16gb-norepack-fit.log").readlines()[-25:]))
else:
    print(f"{'depth':>6} {'pp tok/s':>9} {'tg tok/s':>9} {'VRAM MiB':>9} {'RSS MiB':>8} {'load s':>7}")
    for r in rows:
        if "pp_tps" in r:
            print(f"{r['depth']:>6} {r['pp_tps']['median']:>9.0f} {r['tg_tps']['median']:>9.1f} "
                  f"{r['vram_peak_mib']:>9.0f} {r.get('rss_peak_mib', 0):>8} {r['load_s']:>7.0f}")
        else:
            print("ERROR", r.get("error", r))
    log = open("bench/results/llama-server-colab-t4-12-16gb-norepack-fit.log").read()
    print("\n--- offload / buffers")
    print("\n".join(l for l in log.splitlines() if re.search(r"offload|model buffer size|KV buffer size|fit", l))[-2500:])
    print("\n--- raw JSONL")
    for r in rows:
        print(json.dumps(r))